In [2]:
import os
import matplotlib.pyplot as plt
from vizlib import PALETTE, save_fig, simulate_cvd, style_defaults, verdict, worst_pair

## 1: Import Toolkit and Define Palettes

In [3]:
# Ensure vizlib can be imported from your project structure
style_defaults()

# Choose your custom palette (e.g., matplotlib tab10 first four colors)
my_palette = ["#1F77B4", "#FF7F0E", "#2CA02C", "#D62728"]
my_label = "matplotlib tab10, first four"

# Course palette combining the standard 3 slots plus the 4th slot
course_palette = PALETTE["series"] + [PALETTE["series_4th"]]
kinds = ["normal", "deutan", "protan", "tritan"]

print("Palettes loaded successfully!")

Palettes loaded successfully!


## 2: Generate and Save the Swatch Sheet

In [4]:
def swatch_sheet(palettes, path):
    fig, axes = plt.subplots(len(palettes), len(kinds),
                             figsize=(3.0 * len(kinds), 2.1 * len(palettes)))
    if len(palettes) == 1:
        axes = axes.reshape(1, -1)
    for row, (pal, label) in enumerate(palettes):
        for col, kind in enumerate(kinds):
            ax = axes[row, col]
            shown = pal if kind == "normal" else simulate_cvd(pal, kind)
            for i, hexc in enumerate(shown):
                ax.add_patch(plt.Rectangle((i, 0), 0.92, 1, color=hexc))
                ax.text(i + 0.46, -0.22, f"S{i+1}", ha="center", fontsize=8,
                        color=PALETTE["ink_soft"])
            ax.set_xlim(-0.1, len(pal))
            ax.set_ylim(-0.4, 1.1)
            ax.axis("off")
            ax.set_title(kind, fontsize=10)
        axes[row, 0].text(-0.05, 1.42, label, transform=axes[row, 0].transAxes,
                          fontsize=11, fontweight="bold", color=PALETTE["ink"])
    fig.suptitle("Prove it, do not claim it", x=0.006, ha="left",
                 fontsize=13, fontweight="bold", color=PALETTE["ink"])
    fig.tight_layout(rect=[0, 0, 1, 0.9])
    out = save_fig(fig, path,
                   "Machado, Oliveira & Fernandes (2009) severity-1.0 "
                   "matrices, applied in linear RGB.")
    plt.close(fig)
    return out

# Save the swatch sheet
output_path = swatch_sheet([(my_palette, my_label),
                            (course_palette, "The course palette")],
                           "partC_palette.png")
print("Swatch sheet saved to:", output_path)

Swatch sheet saved to: partC_palette.png


##  3: Audit Palettes Using OKLab Delta-E ($\Delta E$)

In [5]:
def audit(pal, label):
    print(label)
    rows = []
    for kind in kinds:
        i, j, a, b, d = worst_pair(pal, kind)
        rows.append((kind, i, j, a, b, d))
        print(f"  {kind:7s} closest pair  S{i} {a} vs S{j} {b}"
              f"   ΔE {d:5.1f}   {verdict(d)}")
    print()
    return rows

print("--- AUDITING CUSTOM PALETTE ---")
audit(my_palette, my_label)

print("--- AUDITING COURSE PALETTE ---")
audit(course_palette, "The course palette (3 slots + the 4th)")

--- AUDITING CUSTOM PALETTE ---
matplotlib tab10, first four
  normal  closest pair  S2 #FF7F0E vs S4 #D62728   ΔE  18.6   clear
  deutan  closest pair  S3 #2CA02C vs S4 #D62728   ΔE   4.1   FAILS — same colour
  protan  closest pair  S2 #FF7F0E vs S3 #2CA02C   ΔE   0.7   FAILS — same colour
  tritan  closest pair  S1 #1F77B4 vs S3 #2CA02C   ΔE   6.7   needs a second channel

--- AUDITING COURSE PALETTE ---
The course palette (3 slots + the 4th)
  normal  closest pair  S2 #D55E00 vs S4 #CC79A7   ΔE  16.4   clear
  deutan  closest pair  S3 #009E73 vs S4 #CC79A7   ΔE   7.5   needs a second channel
  protan  closest pair  S1 #0072B2 vs S4 #CC79A7   ΔE   9.5   ok
  tritan  closest pair  S1 #0072B2 vs S3 #009E73   ΔE   8.4   ok



[('normal', 2, 4, '#D55E00', '#CC79A7', 16.419658206690205),
 ('deutan', 3, 4, '#009E73', '#CC79A7', 7.546637952916592),
 ('protan', 1, 4, '#0072B2', '#CC79A7', 9.494498011532),
 ('tritan', 1, 3, '#0072B2', '#009E73', 8.372200722672284)]

## Question 3 (Worst Pairs & $\Delta E$):

When you run the audit above, inspect the outputs for the closest pairs. For instance, standard palettes like tab10 typically fail heavily under red-green deficiencies (deutan/protan) because blue/orange or green/red pairs collapse into very low $\Delta E$ values (often under 6, meaning "same color" to a color-blind reader). Look at your printed output to copy the exact slot numbers and $\Delta E$ values into your report.

For the course palette with the 4th slot, the fourth color drops the CVD margin significantly (down to a low $\Delta E$), which triggers a warning flag. 

## Question 4 (Rule for Charts with >3 Categories):

### The Rule: 
Never solve a category overflow by adding a 4th or 5th distinct hue.
### Explanation: 
Because adding more colors drops perceptual contrast ($\Delta E$) below safe thresholds under CVD simulation, any chart requiring more than three categories must rely on a secondary encoding channel instead of color alone (such as direct text labels, distinct marker shapes, patterns, or small multiples/facets).   